# Diabetic Retinopathy — Self-Contained Training Notebook

**Two tracks in a single notebook:**
- **Model 1 (Custom):** SEResNet9 trained from scratch, no pretrained weights.
- **Model 2 (Fine-Tuning):** EfficientNet-B2 + DenseNet-121 multi-scale ensemble with learnable weights.

**Usage:** Set paths in section `# 1`, then `Restart & Run All`.  
No local module imports — all code is defined inline.

**Drive layout expected:**
```
MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/
  data/          <- images/ + train/val/test CSVs
  results/
    checkpoints/custom/     <- Model 1 checkpoint (.pth)
    checkpoints/finetune/   <- Model 2 checkpoint (.pth)
    outputs/                <- prediction CSVs + figures
    Submissions/            <- codabench_submission.zip
```

# 0. Setup & Dependencies

In [ ]:
# Uncomment on first Colab run if packages are missing
# !pip install -q scikit-image opencv-python-headless scikit-learn

import sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

# 1. Global Configuration & Paths

**Edit only this cell** to adapt to your environment.

Expected `DATA_DIR` structure:
```
DATA_DIR/
  images/      <- .jpg files named by 5-digit id
  train.csv    <- columns: id (str), eye (int), label (int 0-4)
  val.csv
  test.csv     <- label = -1 (unlabelled)
```

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy'
DATA_DIR   = os.path.join(DRIVE_BASE, 'data')      # images + CSVs source
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'results')   # checkpoints + CSVs + ZIP

# ── Training flags ─────────────────────────────────────────────────────────────
# Set to False to skip training and load the saved checkpoint from OUTPUT_DIR
TRAIN_MODEL_1 = True
TRAIN_MODEL_2 = True

# ── Custom hyperparameters (Model 1 — SEResNet9) ───────────────────────────────
C_IMG_SIZE      = 512
C_BATCH_SIZE    = 16
C_NUM_EPOCHS    = 60
C_LR            = 5e-4
C_WEIGHT_DECAY  = 5e-4
C_WARMUP_EPOCHS = 5
C_FOCAL_ALPHA   = 0.5   # symmetric: WeightedRandomSampler already balances classes
C_FOCAL_GAMMA   = 2.0
C_DROPOUT       = 0.1
C_EARLY_STOP    = 15
C_CHECKPOINT    = os.path.join(OUTPUT_DIR, 'checkpoints', 'custom', 'seresnet9_best.pth')
C_OUTPUT_CSV    = os.path.join(OUTPUT_DIR, 'outputs', 'predictions_custom.csv')

# ── Fine-tune hyperparameters (Model 2 — Ensemble) ────────────────────────────
FT_IMG_SIZE     = 512
FT_BATCH_SIZE   = 16
FT_NUM_EPOCHS   = 50
FT_LR           = 3e-4
FT_WEIGHT_DECAY = 1e-4
FT_UNFREEZE_N   = 3
FT_FOCAL_ALPHA  = 0.25
FT_FOCAL_GAMMA  = 2.0
FT_LR_PATIENCE  = 3
FT_LR_FACTOR    = 0.3
FT_EARLY_STOP   = 10
FT_ENS_SCALES   = [224, 512]   # one resolution per ensemble member
FT_NUM_TTA      = 10
FT_CHECKPOINT   = os.path.join(OUTPUT_DIR, 'checkpoints', 'finetune', 'ensemble_best.pth')
FT_OUTPUT_CSV   = os.path.join(OUTPUT_DIR, 'outputs', 'predictions_finetune.csv')

# ── Copy data to local disk for faster I/O (recommended on Colab) ─────────────
COPY_DATA_LOCAL = True
LOCAL_DATA_DIR  = '/content/data'

# ── Reproducibility seed ───────────────────────────────────────────────────────
SEED = 42

# ── Create output directories ─────────────────────────────────────────────────
for _d in [
    os.path.join(OUTPUT_DIR, 'checkpoints', 'custom'),
    os.path.join(OUTPUT_DIR, 'checkpoints', 'finetune'),
    os.path.join(OUTPUT_DIR, 'outputs'),
    os.path.join(OUTPUT_DIR, 'Submissions'),
]:
    os.makedirs(_d, exist_ok=True)

print('Config OK.')
print(f'  DATA_DIR   : {DATA_DIR}')
print(f'  OUTPUT_DIR : {OUTPUT_DIR}')
print(f'  Train M1={TRAIN_MODEL_1}  Train M2={TRAIN_MODEL_2}')

In [ ]:
# Copy dataset from Drive to local disk to avoid slow network I/O during training
import shutil

if COPY_DATA_LOCAL and not os.path.exists(LOCAL_DATA_DIR):
    print('Copying dataset to local disk...')
    shutil.copytree(DATA_DIR, LOCAL_DATA_DIR)
    print('Done.')

if COPY_DATA_LOCAL and os.path.exists(LOCAL_DATA_DIR):
    DATA_DIR = LOCAL_DATA_DIR

print(f'Active DATA_DIR: {DATA_DIR}')

# 2. Common Utilities

Imports, seeds, dataset, shared transforms, FocalLoss, WeightedRandomSampler, training loop, metrics — defined **once** and reused by both tracks.

In [ ]:
# ── Standard & third-party imports ────────────────────────────────────────────
import copy
import csv
import random
import time
from zipfile import ZipFile

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from PIL import Image
from skimage import color, io, transform, util
from sklearn import metrics as sk_metrics
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

# ── Global seeds ──────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print(f'Device: {device}  |  AMP: {USE_AMP}')

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class RetinopathyDataset(Dataset):
    """Retinal fundus image dataset for binary DR classification.

    CSV columns: id (5-digit str), eye (0=left, 1=right), label (0-4 or -1 for test).
    Right-eye images are horizontally mirrored before any transform.
    Labels are binarised: (label > 0) -> 1 (DR), 0 (No DR). -1 preserved for test set.
    """
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.data = pd.read_csv(csv_file, header=0,
                                dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=SEED).permutation(len(self.data))
            self.data = self.data.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir  = root_dir
        self.img_dir   = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_path = os.path.join(self.img_dir, self.data.id[idx] + '.jpg')
        image    = io.imread(img_path)
        if self.data.eye[idx] == 1:
            image = image[:, ::-1, :].copy()   # flip right eye; .copy() avoids read-only view
        label  = self.data.label[idx]
        binary = (label > 0).astype(np.int64) if label >= 0 else label
        sample = {'image': image, 'eye': self.data.eye[idx], 'label': binary}
        if self.transform:
            sample = self.transform(sample)
        return sample

In [ ]:
# ── Shared transforms (used by both tracks) ───────────────────────────────────

class CropByEye:
    """Crop the image to the bounding box of the fundus disc, removing black borders."""
    def __init__(self, threshold=0.10, border=1):
        self.threshold = threshold
        self.border    = (border, border) if isinstance(border, int) else tuple(border)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w  = image.shape[:2]
        gray  = color.rgb2gray(image)
        _, mask = cv2.threshold(gray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx  = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = max(sidx[1].min() - self.border[1], 0)
        maxx = min(sidx[1].max() + 1 + self.border[1], w)
        miny = max(sidx[0].min() - self.border[0], 0)
        maxy = min(sidx[0].max() + 1 + self.border[0], h)   # border[0] for row axis
        return {'image': image[miny:maxy, minx:maxx], 'eye': eye, 'label': label}


class Rescale:
    """Resize image. output_size: int (shorter-side rule) or (h, w) tuple."""
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size          if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop:
    """Randomly crop the image to output_size (int for square, or (h, w))."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        return {'image': image[top:top+new_h, left:left+new_w], 'eye': eye, 'label': label}


class CenterCrop:
    """Central crop to output_size (int for square, or (h, w))."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = max((h - new_h) // 2, 0)
        left = max((w - new_w) // 2, 0)
        return {'image': image[top:top+new_h, left:left+new_w], 'eye': eye, 'label': label}


class ToTensor:
    """Convert HxWxC numpy array to CxHxW float32 tensor."""
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)).copy()).float()
        return {'image': image, 'eye': eye, 'label': torch.tensor(label, dtype=torch.long)}


# ── torchvision wrappers (numpy -> PIL -> TV transform -> numpy) ───────────────
def _to_pil(img):  return Image.fromarray(util.img_as_ubyte(img))
def _to_np(pil):   return util.img_as_float(np.asarray(pil))

class TVRandomHorizontalFlip:
    def __init__(self, p=0.5): self._t = transforms.RandomHorizontalFlip(p=p)
    def __call__(self, s):
        return {'image': _to_np(self._t(_to_pil(s['image']))), 'eye': s['eye'], 'label': s['label']}

class TVRandomRotation:
    def __init__(self, deg): self._t = transforms.RandomRotation(deg)
    def __call__(self, s):
        return {'image': _to_np(self._t(_to_pil(s['image']))), 'eye': s['eye'], 'label': s['label']}

class TVColorJitter:
    def __init__(self, brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0):
        self._t = transforms.ColorJitter(brightness=brightness, contrast=contrast,
                                          saturation=saturation, hue=hue)
    def __call__(self, s):
        return {'image': _to_np(self._t(_to_pil(s['image']))), 'eye': s['eye'], 'label': s['label']}

class TVNormalize:
    def __init__(self, mean, std): self._t = transforms.Normalize(mean=mean, std=std)
    def __call__(self, s):
        return {'image': self._t(s['image']), 'eye': s['eye'], 'label': s['label']}

# ImageNet statistics for pretrained backbones
_IN_MEAN = [0.485, 0.456, 0.406]
_IN_STD  = [0.229, 0.224, 0.225]

In [ ]:
# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Binary Focal Loss (Lin et al., ICCV 2017).

    Down-weights easy/well-classified examples exponentially, forcing the
    optimiser to concentrate gradient on hard boundary cases (e.g. mild DR
    with a single faint microaneurysm). Receives raw logits.

    Args:
        gamma: focusing parameter (2.0 recommended).
        alpha: positive-class weight in [0, 1].
    """
    def __init__(self, gamma=2.0, alpha=0.5, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        loss    = alpha_t * (1.0 - p_t) ** self.gamma * bce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss

In [ ]:
# ── Generic training loop ─────────────────────────────────────────────────────
def train_model(model, criterion, optimizer, scheduler, dataloaders,
                dataset_sizes, device, num_epochs=25, temp_save_path=None,
                use_amp=False, early_stop_patience=10, scheduler_mode='epoch'):
    """Train and validate a model, keeping the best checkpoint.

    scheduler_mode:
      'epoch'   -> scheduler.step() once per epoch (SequentialLR / CosineAnnealing)
      'plateau' -> scheduler.step(val_auc) (ReduceLROnPlateau)

    The model receives raw LOGITS from the network. FocalLoss calls
    binary_cross_entropy_with_logits internally — never pass probabilities here.

    Returns (model, best_auc, best_epoch, elapsed_seconds).
    """
    since  = time.time()
    scaler = GradScaler(device=device.type, enabled=use_amp)

    best_weights      = copy.deepcopy(model.state_dict())
    best_auc          = 0.0
    best_epoch        = -1
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs-1}  ', end='')

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            n        = dataset_sizes[phase]
            all_out  = np.zeros(n, dtype=float)
            all_lbl  = np.zeros(n, dtype=int)
            run_loss = 0.0
            cursor   = 0

            for sample in dataloaders[phase]:
                inputs = sample['image'].to(device).float()
                labels = sample['label'].to(device).float()
                bs     = labels.shape[0]
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    with autocast(device_type=device.type, enabled=use_amp):
                        outputs = model(inputs)          # raw logits from both models
                    logits = outputs.flatten().float()   # (N,) logit for FocalLoss
                    loss   = criterion(logits, labels)
                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                run_loss += loss.item() * bs
                # Collect probabilities for AUC (sigmoid applied here, not inside model)
                probs = torch.sigmoid(outputs.detach()).flatten().cpu().numpy()
                all_out[cursor:cursor+bs] = probs
                all_lbl[cursor:cursor+bs] = labels.cpu().numpy().astype(int)
                cursor += bs

            if phase == 'train' and scheduler_mode == 'epoch':
                scheduler.step()

            epoch_loss = run_loss / n
            epoch_auc  = sk_metrics.roc_auc_score(all_lbl, all_out)
            lr         = optimizer.param_groups[0]['lr']
            print(f'{phase} loss={epoch_loss:.4f} AUC={epoch_auc:.4f} lr={lr:.1e}  ', end='')

            if phase == 'val':
                if scheduler_mode == 'plateau':
                    scheduler.step(epoch_auc)
                if epoch_auc > best_auc:
                    best_auc, best_epoch = epoch_auc, epoch
                    epochs_no_improve    = 0
                    best_weights = copy.deepcopy(model.state_dict())
                    if temp_save_path:
                        os.makedirs(os.path.dirname(temp_save_path), exist_ok=True)
                        torch.save(model.state_dict(), temp_save_path)
                    print('[BEST]', end='')
                else:
                    epochs_no_improve += 1
        print()

        if early_stop_patience and epochs_no_improve >= early_stop_patience:
            print(f'Early stopping at epoch {epoch} ({early_stop_patience} epochs without improvement)')
            break

    elapsed = time.time() - since
    print(f'\nTraining complete: {elapsed//60:.0f}m {elapsed%60:.0f}s')
    print(f'Best val AUC: {best_auc:.4f} at epoch {best_epoch}')
    model.load_state_dict(best_weights)
    return model, best_auc, best_epoch, elapsed

In [ ]:
# ── Inference & output helpers ────────────────────────────────────────────────
def predict_test(model, test_loader, device):
    """Run inference on test set. Returns (N,1) numpy array of DR scores in [0,1].

    Applies sigmoid to convert raw logits to probabilities.
    Use for SEResNet9 (Custom track).
    """
    model.eval()
    preds = []
    with torch.no_grad():
        for sample in test_loader:
            out = torch.sigmoid(model(sample['image'].to(device).float()))
            preds.append(out.view(-1, 1).cpu().numpy())
    return np.concatenate(preds, axis=0)


def save_csv(scores, path):
    """Save (N,1) score array as one float per line — Codabench format."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as f:
        csv.writer(f).writerows(scores.tolist())
    print(f'CSV saved -> {path}  ({len(scores)} rows)')


def make_submission_zip(custom_csv, finetune_csv, output_dir):
    """Build codabench_submission.zip containing output_custom.csv and output_ft.csv."""
    subs_dir    = os.path.join(output_dir, 'Submissions')
    os.makedirs(subs_dir, exist_ok=True)
    custom_dest = os.path.join(subs_dir, 'output_custom.csv')
    ft_dest     = os.path.join(subs_dir, 'output_ft.csv')
    shutil.copy2(custom_csv, custom_dest)
    shutil.copy2(finetune_csv, ft_dest)
    zip_path = os.path.join(subs_dir, 'codabench_submission.zip')
    with ZipFile(zip_path, 'w') as zf:
        zf.write(custom_dest,  arcname='output_custom.csv')
        zf.write(ft_dest,      arcname='output_ft.csv')
    print(f'Submission ZIP -> {zip_path}')
    return zip_path


def make_balanced_sampler(csv_path):
    """Build WeightedRandomSampler that produces 50/50 positive/negative batches.

    Reads labels directly from the CSV (no image decoding).
    """
    df     = pd.read_csv(csv_path, dtype={'id': str, 'eye': int, 'label': int})
    labels = (df['label'] > 0).astype(int).tolist()
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    w_map  = {0: 1.0 / n_neg, 1: 1.0 / n_pos}
    weights = [w_map[l] for l in labels]
    print(f'Sampler: {n_pos} pos / {n_neg} neg -> balanced batches')
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# 3. Model 1 — Custom (SEResNet9)

> **Evaluator rule:** Zero pretrained weights, zero torchvision/timm architecture classes.  
> SEResNet9 is fully defined here using only `torch.nn` primitives.

## 3.1 Architecture

In [ ]:
# ── Internal helpers ──────────────────────────────────────────────────────────
def _conv_bn_relu(in_ch, out_ch, **kw):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, bias=False, **kw),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class _ResBlock(nn.Module):
    """Two Conv3x3+BN+ReLU layers with an identity skip connection."""
    def __init__(self, ch):
        super().__init__()
        self.net  = nn.Sequential(
            _conv_bn_relu(ch, ch, kernel_size=3, padding=1),
            nn.Conv2d(ch, ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(x + self.net(x))

class _SEBlock(nn.Module):
    """Squeeze-and-Excitation block: channel-wise attention."""
    def __init__(self, ch, reduction=16):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.exc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, ch, bias=False), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.exc(x).view(x.size(0), x.size(1), 1, 1)


# ── SEResNet9 — Custom track primary model ────────────────────────────────────
class SEResNet9(nn.Module):
    """ResNet-9 with Squeeze-and-Excitation attention blocks. ~1.65M parameters.

    Spatial flow at 512x512:
      prep:   3  -> 32  ch   512x512
      layer1: 32 -> 64  ch   256x256  (MaxPool)
      res1 + SE(64)          256x256
      layer2: 64 -> 128 ch   128x128  (MaxPool)
      layer3: 128-> 256 ch    64x64   (MaxPool)
      res2 + SE(256)          64x64
      GAP -> Dropout(p) -> Linear(256, 1)  -- raw logit

    Key design decisions:
    - SE blocks: channel attention amplifies the green channel (microaneurysms)
      without extra spatial compute.
    - Kaiming init: stabilises gradients from random initialisation.
    - Output: raw logit (N, 1). Apply sigmoid externally at inference time.
    """
    def __init__(self, in_channels=3, dropout=0.1):
        super().__init__()
        self.prep   = _conv_bn_relu(in_channels, 32, kernel_size=3, padding=1)
        self.layer1 = nn.Sequential(_conv_bn_relu(32, 64, kernel_size=3, padding=1), nn.MaxPool2d(2, 2))
        self.res1   = _ResBlock(64)
        self.se1    = _SEBlock(64)
        self.layer2 = nn.Sequential(_conv_bn_relu(64, 128, kernel_size=3, padding=1), nn.MaxPool2d(2, 2))
        self.layer3 = nn.Sequential(_conv_bn_relu(128, 256, kernel_size=3, padding=1), nn.MaxPool2d(2, 2))
        self.res2   = _ResBlock(256)
        self.se2    = _SEBlock(256)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, 1))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.prep(x)
        x = self.se1(self.res1(self.layer1(x)))
        x = self.se2(self.res2(self.layer3(self.layer2(x))))
        return self.head(self.gap(x).flatten(1))   # (N, 1) raw logit


# Quick sanity check
_m = SEResNet9()
_n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'SEResNet9: {_n:,} trainable parameters')
with torch.no_grad():
    _out = _m(torch.randn(2, 3, 512, 512))
print(f'Output shape: {_out.shape}  (expected: (2, 1))')
assert _out.shape == (2, 1)
# Confirm raw logit (not sigmoid probability): at least some values outside [0,1]
assert ((_out < 0) | (_out > 1)).any(), 'Output looks like sigmoid was applied internally'
print('Architecture check passed.')
del _m, _out

## 3.2 Custom Transforms + Dataset & DataLoaders

In [ ]:
# ── Custom-track-specific transforms ─────────────────────────────────────────

class BenGraham:
    """Ben Graham fundus illumination normalisation.

    cv2.addWeighted(img, 4, GaussBlur(img, sigma), -4, 128)
    Maps the background to neutral grey (128) and maximises contrast of lesions.

    IMPORTANT: Must be applied AFTER CropByEye. BenGraham maps black borders to
    grey (~128), which breaks CropByEye's brightness-threshold eye detection.
    """
    def __init__(self, sigma=15.0):
        self.sigma = sigma

    def __call__(self, sample):
        img, eye, lbl = sample['image'], sample['eye'], sample['label']
        if img.dtype != np.uint8:
            img = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        blurred = cv2.GaussianBlur(img, (0, 0), self.sigma)
        img = np.clip(cv2.addWeighted(img, 4, blurred, -4, 128), 0, 255).astype(np.uint8)
        return {'image': img, 'eye': eye, 'label': lbl}


class RandomCutout:
    """Erase n_holes random patches of patch_size x patch_size pixels.

    Forces the network to learn features distributed across the whole retina
    instead of relying on a single lesion location.
    Apply before ToTensor on float images in [0, 1].
    """
    def __init__(self, n_holes=2, patch_size=32, fill_value=0.5):
        self.n_holes = n_holes; self.patch_size = patch_size; self.fill_value = fill_value

    def __call__(self, sample):
        img, eye, lbl = sample['image'], sample['eye'], sample['label']
        h, w = img.shape[:2]; half = self.patch_size // 2
        img = img.copy()
        for _ in range(self.n_holes):
            cy, cx = np.random.randint(0, h), np.random.randint(0, w)
            img[max(cy-half,0):min(cy+half,h), max(cx-half,0):min(cx+half,w)] = self.fill_value
        return {'image': img, 'eye': eye, 'label': lbl}


class PerImageNormalize:
    """Per-channel zero-mean unit-std normalisation applied to a float32 tensor.

    Eliminates domain bias vs. ImageNet statistics — retinal image channel
    distributions are very different from natural images.
    Apply after ToTensor.
    """
    def __call__(self, sample):
        img, eye, lbl = sample['image'], sample['eye'], sample['label']
        mean = img.mean(dim=(1, 2), keepdim=True)
        std  = img.std(dim=(1, 2),  keepdim=True).clamp(min=1e-6)
        return {'image': (img - mean) / std, 'eye': eye, 'label': lbl}


def get_custom_train_transforms(img_size=512):
    """Training pipeline for the Custom track (SEResNet9)."""
    scale_size = int(img_size * 256 / 224)
    return transforms.Compose([
        CropByEye(0.10, 1),                                             # 1) remove black borders
        BenGraham(sigma=15.0),                                          # 2) illumination normalisation (AFTER CropByEye)
        Rescale(scale_size),
        RandomCrop(img_size),
        TVRandomHorizontalFlip(p=0.5),
        TVRandomRotation(180),                                          # retina has full rotational symmetry
        TVColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0),  # no hue: colour encodes lesion type
        RandomCutout(n_holes=2, patch_size=32, fill_value=0.5),
        ToTensor(),
        PerImageNormalize(),
    ])


def get_custom_val_transforms(img_size=512):
    """Deterministic validation/test pipeline for the Custom track."""
    scale_size = int(img_size * 256 / 224)
    return transforms.Compose([
        CropByEye(0.10, 1),
        BenGraham(sigma=15.0),
        Rescale(scale_size),
        CenterCrop(img_size),
        ToTensor(),
        PerImageNormalize(),
    ])

In [ ]:
# ── Custom DataLoaders ────────────────────────────────────────────────────────
# num_workers=2: safer on Colab (reduce to 0 if DataLoader hangs)
c_train_ds = RetinopathyDataset(os.path.join(DATA_DIR, 'train.csv'), DATA_DIR,
                                transform=get_custom_train_transforms(C_IMG_SIZE))
c_val_ds   = RetinopathyDataset(os.path.join(DATA_DIR, 'val.csv'),   DATA_DIR,
                                transform=get_custom_val_transforms(C_IMG_SIZE))
c_test_ds  = RetinopathyDataset(os.path.join(DATA_DIR, 'test.csv'),  DATA_DIR,
                                transform=get_custom_val_transforms(C_IMG_SIZE))

c_sampler = make_balanced_sampler(os.path.join(DATA_DIR, 'train.csv'))

c_train_loader = DataLoader(c_train_ds, batch_size=C_BATCH_SIZE, sampler=c_sampler,
                            num_workers=2, pin_memory=True)
c_val_loader   = DataLoader(c_val_ds,   batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)
c_test_loader  = DataLoader(c_test_ds,  batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)

print(f'Custom — Train: {len(c_train_ds)} | Val: {len(c_val_ds)} | Test: {len(c_test_ds)}')

## 3.3 Training

In [ ]:
if TRAIN_MODEL_1:
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    c_model = SEResNet9(in_channels=3, dropout=C_DROPOUT).to(device)
    c_crit  = FocalLoss(gamma=C_FOCAL_GAMMA, alpha=C_FOCAL_ALPHA)
    c_opt   = optim.Adam(c_model.parameters(), lr=C_LR, weight_decay=C_WEIGHT_DECAY)

    # SequentialLR: linear warmup (epochs 0-4, factor 0.01->1.0) then CosineAnnealing
    # Warmup prevents collapse toward the majority class during random initialisation
    _warmup = optim.lr_scheduler.LinearLR(
        c_opt, start_factor=0.01, end_factor=1.0, total_iters=C_WARMUP_EPOCHS)
    _cosine = optim.lr_scheduler.CosineAnnealingLR(
        c_opt, T_max=C_NUM_EPOCHS - C_WARMUP_EPOCHS, eta_min=1e-6)
    c_sched = optim.lr_scheduler.SequentialLR(
        c_opt, schedulers=[_warmup, _cosine], milestones=[C_WARMUP_EPOCHS])

    c_model, c_best_auc, c_best_epoch, c_elapsed = train_model(
        c_model, c_crit, c_opt, c_sched,
        dataloaders   = {'train': c_train_loader, 'val': c_val_loader},
        dataset_sizes = {'train': len(c_train_ds), 'val': len(c_val_ds)},
        device        = device,
        num_epochs    = C_NUM_EPOCHS,
        temp_save_path= C_CHECKPOINT,
        use_amp       = USE_AMP,
        early_stop_patience = C_EARLY_STOP,
        scheduler_mode= 'epoch',
    )
    print(f'Model 1 saved: {C_CHECKPOINT}')
else:
    print(f'TRAIN_MODEL_1=False — loading checkpoint from {C_CHECKPOINT}')
    c_model = SEResNet9(in_channels=3, dropout=C_DROPOUT).to(device)
    c_model.load_state_dict(torch.load(C_CHECKPOINT, map_location=device, weights_only=True))
    c_best_auc, c_best_epoch, c_elapsed = None, None, 0
    print('Checkpoint loaded.')

## 3.4 Evaluation & CSV Generation

In [ ]:
# Validation predictions (for ROC curve)
c_model.eval()
c_val_preds, c_val_labels = [], []
with torch.no_grad():
    for sample in c_val_loader:
        probs = torch.sigmoid(c_model(sample['image'].to(device).float())).view(-1).cpu().numpy()
        c_val_preds.extend(probs)
        c_val_labels.extend(sample['label'].numpy())

c_val_preds  = np.array(c_val_preds)
c_val_labels = np.array(c_val_labels)
c_val_auc    = sk_metrics.roc_auc_score(c_val_labels, c_val_preds)
print(f'Val AUC — Model 1 (Custom SEResNet9): {c_val_auc:.4f}')

# Test predictions (sigmoid applied inside predict_test)
c_test_scores = predict_test(c_model, c_test_loader, device)
assert c_test_scores.shape == (len(c_test_ds), 1), f'Unexpected shape: {c_test_scores.shape}'
assert c_test_scores.min() >= 0.0 and c_test_scores.max() <= 1.0, 'Scores outside [0,1]'
save_csv(c_test_scores, C_OUTPUT_CSV)

# 4. Model 2 — Fine-Tuning Ensemble

EfficientNet-B2 + DenseNet-121 with multi-scale input (224 px / 512 px) and learnable Softmax ensemble weights.

## 4.1 Architecture

In [ ]:
# ── BaseModel: single pretrained backbone with replaced head ──────────────────
class BaseModel(nn.Module):
    """Pretrained backbone fine-tuned for binary DR classification.

    Supported backbones: 'efficientnet_b2', 'densenet121', 'mobilenet_v3_large'.
    Head is replaced with a single Linear(in_features, 1) — raw logit output.

    Args:
        backbone_name: name string for the backbone.
        unfreeze_n:    number of layer groups (from the end) to unfreeze.
        pretrained:    load ImageNet weights.
    """
    def __init__(self, backbone_name='efficientnet_b2', unfreeze_n=3, pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name
        self.model = self._build(backbone_name, pretrained)
        self._freeze_and_unfreeze(unfreeze_n)
        total     = sum(p.numel() for p in self.model.parameters())
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f'  [{backbone_name}] total={total:,}  trainable={trainable:,}')

    def _build(self, name, pretrained):
        w = 'IMAGENET1K_V1' if pretrained else None
        n = name.lower()
        if n == 'efficientnet_b2':
            m = models.efficientnet_b2(weights=w)
            m.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True),
                                          nn.Linear(m.classifier[1].in_features, 1))
        elif n == 'densenet121':
            m = models.densenet121(weights=w)
            m.classifier = nn.Linear(m.classifier.in_features, 1)
        elif n == 'mobilenet_v3_large':
            m = models.mobilenet_v3_large(weights=w)
            m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)
        else:
            raise ValueError(f'Unsupported backbone: {name}')
        return m

    def _layer_groups(self):
        n = self.backbone_name.lower()
        if 'efficientnet' in n:
            return [self.model.features[i] for i in range(len(self.model.features))] + [self.model.classifier]
        elif 'densenet' in n:
            f = self.model.features
            return [f.conv0, f.denseblock1, f.denseblock2, f.denseblock3, f.denseblock4, self.model.classifier]
        elif 'mobilenet' in n:
            return [self.model.features, self.model.classifier]
        return list(self.model.children())

    def _freeze_and_unfreeze(self, n):
        if n == -1:
            return
        for p in self.model.parameters():
            p.requires_grad = False
        for g in self._layer_groups()[-n:]:
            for p in g.parameters():
                p.requires_grad = True

    def forward(self, x):
        return self.model(x)   # (N, 1) raw logit

    def unfreeze_all(self):
        for p in self.model.parameters():
            p.requires_grad = True


# ── EnsembleModel: heterogeneous multi-scale soft-voting ─────────────────────
class EnsembleModel(nn.Module):
    """Multi-scale heterogeneous soft-voting ensemble with learnable Softmax weights.

    Each member receives the input interpolated to a different resolution:
      Member 0 -> scales[0]  (global context)
      Member 1 -> scales[1]  (fine lesions)

    ens_weights are nn.Parameter; Softmax ensures they sum to 1.

    IMPORTANT — logit convention:
      forward() returns a weighted average of RAW LOGITS (no sigmoid).
      This keeps the output compatible with FocalLoss (binary_cross_entropy_with_logits).
      Sigmoid is applied externally: at inference time and inside predict_tta().
    """
    def __init__(self, backbone_names=None, unfreeze_n=3, pretrained=True,
                 ensemble_scales=None, num_tta=10):
        super().__init__()
        if backbone_names is None:
            backbone_names = ['efficientnet_b2', 'densenet121']
        self.ensemble_scales = ensemble_scales or [224, 512]
        self.num_tta         = num_tta

        members = []
        for spec in backbone_names:
            if isinstance(spec, nn.Module):
                members.append(spec)
            else:
                members.append(BaseModel(spec, unfreeze_n=unfreeze_n, pretrained=pretrained))

        self.members_list = nn.ModuleList(members)
        self.ens_weights  = nn.Parameter(torch.zeros(len(members), dtype=torch.float32))

        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'[EnsembleModel] Backbones : {backbone_names}')
        print(f'[EnsembleModel] Scales    : {self.ensemble_scales}')
        print(f'[EnsembleModel] Trainable : {total:,}')

    def forward(self, x):
        """Weighted average of member logits. Returns (N,1) RAW LOGIT."""
        w        = torch.softmax(self.ens_weights, dim=0)
        logit_out = torch.zeros(x.size(0), 1, device=x.device, dtype=x.dtype)
        for i, member in enumerate(self.members_list):
            res  = self.ensemble_scales[i % len(self.ensemble_scales)]
            x_r  = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            logit_out = logit_out + w[i] * member(x_r)   # member(x_r) is a logit
        return logit_out   # raw logit — compatible with FocalLoss

    @torch.no_grad()
    def predict_tta(self, x, n_passes=None):
        """N stochastic forward passes with random flips/rotations. Returns mean PROBABILITY."""
        n_passes = n_passes or self.num_tta
        self.eval()
        prob_sum = torch.zeros(x.size(0), 1, device=x.device)
        for _ in range(n_passes):
            x_aug = x.clone()
            if torch.rand(1).item() > 0.5: x_aug = torch.flip(x_aug, dims=[3])
            if torch.rand(1).item() > 0.5: x_aug = torch.flip(x_aug, dims=[2])
            k = torch.randint(0, 4, (1,)).item()
            if k > 0: x_aug = torch.rot90(x_aug, k=k, dims=[2, 3])
            prob_sum = prob_sum + torch.sigmoid(self.forward(x_aug))   # sigmoid HERE
        return prob_sum / n_passes   # mean probability in [0,1]

## 4.2 Fine-Tune Transforms + Dataset & DataLoaders

In [ ]:
# ── Fine-tune-track-specific transforms ───────────────────────────────────────

def _clahe_channel(image, ch, clip=2.0, grid=(4, 4)):
    """Apply CLAHE to channel `ch` of an RGB image (uint8 or float[0,1])."""
    is_float = image.dtype != np.uint8
    img_u8   = util.img_as_ubyte(np.clip(image, 0, 1)) if is_float else image.copy()
    clahe    = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid)
    out      = img_u8.copy()
    out[:, :, ch] = clahe.apply(img_u8[:, :, ch])
    return util.img_as_float(out) if is_float else out


class DualChannelEnhancement:
    """Masked CLAHE applied independently to the green (ch=1) and red (ch=0) channels.

    Medical rationale:
    - Green channel: highest contrast for microaneurysms and hard exudates.
    - Red channel: highlights haemorrhages and neovascularisation.
    - Fundus mask: preserves black borders and prevents amplifying noise outside the disc.
    """
    def __init__(self, mask_threshold=0.05, clip_limit=2.0, tile_grid=(4, 4)):
        self.thr  = mask_threshold
        self.clip = clip_limit
        self.grid = tile_grid

    def __call__(self, sample):
        img, eye, lbl = sample['image'], sample['eye'], sample['label']
        mask = (color.rgb2gray(img) > self.thr).astype(np.float64)
        img  = _clahe_channel(img, ch=1, clip=self.clip, grid=self.grid)  # green
        img  = _clahe_channel(img, ch=0, clip=self.clip, grid=self.grid)  # red
        img  = img * mask[:, :, np.newaxis]                               # re-apply fundus mask
        return {'image': img, 'eye': eye, 'label': lbl}


class GaussianNoise:
    """Additive zero-mean Gaussian noise to simulate medical sensor variability."""
    def __init__(self, std=0.02):
        self.std = std

    def __call__(self, sample):
        img, eye, lbl = sample['image'], sample['eye'], sample['label']
        noise = np.random.normal(0, self.std, img.shape).astype(img.dtype)
        return {'image': np.clip(img + noise, 0, 1), 'eye': eye, 'label': lbl}


def get_ft_train_transforms(img_size=512):
    """Training pipeline for the Fine-Tune track (EnsembleModel)."""
    scale_size = int(img_size * 256 / 224)
    return transforms.Compose([
        CropByEye(0.10, 1),
        Rescale(scale_size),
        DualChannelEnhancement(mask_threshold=0.05),
        RandomCrop(img_size),
        TVRandomHorizontalFlip(p=0.5),
        TVRandomRotation(30),
        TVColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        GaussianNoise(std=0.02),
        ToTensor(),
        TVNormalize(mean=_IN_MEAN, std=_IN_STD),
    ])


def get_ft_val_transforms(img_size=512):
    """Deterministic validation/test pipeline for the Fine-Tune track."""
    scale_size = int(img_size * 256 / 224)
    return transforms.Compose([
        CropByEye(0.10, 1),
        Rescale(scale_size),
        DualChannelEnhancement(mask_threshold=0.05),
        CenterCrop(img_size),
        ToTensor(),
        TVNormalize(mean=_IN_MEAN, std=_IN_STD),
    ])

In [ ]:
# ── Fine-Tune DataLoaders ─────────────────────────────────────────────────────
# num_workers=2: safer on Colab (reduce to 0 if DataLoader hangs)
ft_train_ds = RetinopathyDataset(os.path.join(DATA_DIR, 'train.csv'), DATA_DIR,
                                 transform=get_ft_train_transforms(FT_IMG_SIZE))
ft_val_ds   = RetinopathyDataset(os.path.join(DATA_DIR, 'val.csv'),   DATA_DIR,
                                 transform=get_ft_val_transforms(FT_IMG_SIZE))
ft_test_ds  = RetinopathyDataset(os.path.join(DATA_DIR, 'test.csv'),  DATA_DIR,
                                 transform=get_ft_val_transforms(FT_IMG_SIZE))

ft_sampler = make_balanced_sampler(os.path.join(DATA_DIR, 'train.csv'))

ft_train_loader = DataLoader(ft_train_ds, batch_size=FT_BATCH_SIZE, sampler=ft_sampler,
                             num_workers=2, pin_memory=True)
ft_val_loader   = DataLoader(ft_val_ds,   batch_size=32, shuffle=False,
                             num_workers=2, pin_memory=True)
ft_test_loader  = DataLoader(ft_test_ds,  batch_size=32, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f'Fine-Tune — Train: {len(ft_train_ds)} | Val: {len(ft_val_ds)} | Test: {len(ft_test_ds)}')

## 4.3 Training

In [ ]:
if TRAIN_MODEL_2:
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    ft_model = EnsembleModel(
        backbone_names  = ['efficientnet_b2', 'densenet121'],
        unfreeze_n      = FT_UNFREEZE_N,
        pretrained      = True,
        ensemble_scales = FT_ENS_SCALES,
        num_tta         = FT_NUM_TTA,
    ).to(device)

    ft_crit  = FocalLoss(gamma=FT_FOCAL_GAMMA, alpha=FT_FOCAL_ALPHA)
    ft_opt   = optim.AdamW(
        filter(lambda p: p.requires_grad, ft_model.parameters()),
        lr=FT_LR, weight_decay=FT_WEIGHT_DECAY,
    )
    # ReduceLROnPlateau monitors val AUC (mode='max')
    ft_sched = optim.lr_scheduler.ReduceLROnPlateau(
        ft_opt, mode='max', factor=FT_LR_FACTOR, patience=FT_LR_PATIENCE
    )

    ft_model, ft_best_auc, ft_best_epoch, ft_elapsed = train_model(
        ft_model, ft_crit, ft_opt, ft_sched,
        dataloaders   = {'train': ft_train_loader, 'val': ft_val_loader},
        dataset_sizes = {'train': len(ft_train_ds), 'val': len(ft_val_ds)},
        device        = device,
        num_epochs    = FT_NUM_EPOCHS,
        temp_save_path= FT_CHECKPOINT,
        use_amp       = USE_AMP,
        early_stop_patience = FT_EARLY_STOP,
        scheduler_mode= 'plateau',
    )
    print(f'Model 2 saved: {FT_CHECKPOINT}')
else:
    print(f'TRAIN_MODEL_2=False — loading checkpoint from {FT_CHECKPOINT}')
    ft_model = EnsembleModel(
        backbone_names  = ['efficientnet_b2', 'densenet121'],
        unfreeze_n      = FT_UNFREEZE_N,
        pretrained      = False,   # weights are loaded from checkpoint
        ensemble_scales = FT_ENS_SCALES,
        num_tta         = FT_NUM_TTA,
    ).to(device)
    ft_model.load_state_dict(torch.load(FT_CHECKPOINT, map_location=device, weights_only=True))
    ft_best_auc, ft_best_epoch, ft_elapsed = None, None, 0
    print('Checkpoint loaded.')

## 4.4 Inference with TTA & CSV Generation

In [ ]:
# Validation predictions (for ROC curve) — apply sigmoid to convert logit -> prob
ft_model.eval()
ft_val_preds, ft_val_labels = [], []
with torch.no_grad():
    for sample in ft_val_loader:
        logits = ft_model(sample['image'].to(device).float())
        probs  = torch.sigmoid(logits).view(-1).cpu().numpy()
        ft_val_preds.extend(probs)
        ft_val_labels.extend(sample['label'].numpy())

ft_val_preds  = np.array(ft_val_preds)
ft_val_labels = np.array(ft_val_labels)
ft_val_auc    = sk_metrics.roc_auc_score(ft_val_labels, ft_val_preds)
print(f'Val AUC — Model 2 (Fine-Tune Ensemble): {ft_val_auc:.4f}')

# Test predictions with TTA (predict_tta applies sigmoid internally)
print(f'Running TTA ({FT_NUM_TTA} passes) on test set...')
ft_model.eval()
ft_test_preds = []
with torch.no_grad():
    for sample in ft_test_loader:
        probs = ft_model.predict_tta(sample['image'].to(device).float(),
                                      n_passes=FT_NUM_TTA).view(-1, 1).cpu().numpy()
        ft_test_preds.append(probs)

ft_test_scores = np.concatenate(ft_test_preds, axis=0)
assert ft_test_scores.shape == (len(ft_test_ds), 1), f'Unexpected shape: {ft_test_scores.shape}'
assert ft_test_scores.min() >= 0.0 and ft_test_scores.max() <= 1.0, 'Scores outside [0,1]'
save_csv(ft_test_scores, FT_OUTPUT_CSV)

# 5. Outputs — Results Summary

In [ ]:
# Build Codabench submission ZIP
zip_path = make_submission_zip(C_OUTPUT_CSV, FT_OUTPUT_CSV, OUTPUT_DIR)
print(f'\nZIP ready for Codabench: {zip_path}')

In [ ]:
# Text summary
def _fmt(secs):
    s = int(secs)
    return f'{s//3600}h {(s%3600)//60}m {s%60}s'

print('=' * 60)
print('  FINAL SUMMARY')
print('=' * 60)
print(f'  Model 1 — Custom (SEResNet9)')
print(f'    Val AUC    : {c_val_auc:.4f}')
print(f'    Best epoch : {c_best_epoch}')
print(f'    Train time : {_fmt(c_elapsed)}')
print(f'    Test CSV   : {C_OUTPUT_CSV}')
print()
print(f'  Model 2 — Fine-Tune Ensemble')
print(f'    Val AUC    : {ft_val_auc:.4f}')
print(f'    Best epoch : {ft_best_epoch}')
print(f'    Train time : {_fmt(ft_elapsed)}')
print(f'    Test CSV   : {FT_OUTPUT_CSV}')
print()
print(f'  Submission ZIP: {zip_path}')
print('=' * 60)

In [ ]:
# Figure 1 — ROC curves (both models)
fig, ax = plt.subplots(figsize=(6, 5))
for preds, labels, name, col in [
    (c_val_preds,  c_val_labels,  f'Custom — SEResNet9 (AUC={c_val_auc:.3f})',   'steelblue'),
    (ft_val_preds, ft_val_labels, f'Fine-Tune Ensemble  (AUC={ft_val_auc:.3f})', 'tomato'),
]:
    fpr, tpr, _ = sk_metrics.roc_curve(labels, preds)
    ax.plot(fpr, tpr, label=name, color=col, lw=2)

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.50)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Validation Set')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'outputs', 'roc_curve.png'), dpi=150)
plt.show()
print('ROC curve saved.')

In [ ]:
# Figure 2 — Prediction score distribution by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for preds, labels, title, ax in [
    (c_val_preds,  c_val_labels,  'Model 1 — Custom SEResNet9',   axes[0]),
    (ft_val_preds, ft_val_labels, 'Model 2 — Fine-Tune Ensemble', axes[1]),
]:
    ax.hist(preds[labels == 0], bins=40, alpha=0.65, label='No DR (label=0)',
            color='steelblue', density=True)
    ax.hist(preds[labels == 1], bins=40, alpha=0.65, label='DR (label=1)',
            color='tomato',    density=True)
    ax.set_xlabel('Predicted score'); ax.set_ylabel('Density')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Prediction Score Distribution by Class — Validation Set', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'outputs', 'pred_distribution.png'), dpi=150,
            bbox_inches='tight')
plt.show()
print('Prediction distribution saved.')

In [ ]:
# Figure 3 — Visual example of transforms
_raw_ds  = RetinopathyDataset(os.path.join(DATA_DIR, 'val.csv'), DATA_DIR, transform=None)
_s       = _raw_ds[0]
_img_raw = _s['image'].copy()
if _s['eye'] == 1:
    _img_raw = _img_raw[:, ::-1, :].copy()   # .copy() avoids read-only view

_crop  = CropByEye(0.10, 1)({'image': _img_raw.copy(), 'eye': 0, 'label': 0})['image']
_ben   = BenGraham(15.0)({'image': _crop.copy(), 'eye': 0, 'label': 0})['image']
_rescaled = Rescale(int(512*256/224))({'image': _crop.copy(), 'eye': 0, 'label': 0})['image']
_dual  = DualChannelEnhancement()({'image': _rescaled.copy(), 'eye': 0, 'label': 0})['image']
_ben_f = _ben.astype(np.float32) / 255.0 if _ben.dtype == np.uint8 else _ben
_cut   = RandomCutout(n_holes=2, patch_size=64, fill_value=0.5)(
    {'image': _ben_f.copy(), 'eye': 0, 'label': 0})['image']

def _show(ax, img, title):
    arr = np.clip(img.astype(np.float32) / 255.0 if img.dtype == np.uint8 else img, 0, 1)
    ax.imshow(arr); ax.set_title(title, fontsize=9); ax.axis('off')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
_show(axes[0], _img_raw,  '(a) Original')
_show(axes[1], _ben,      '(b) BenGraham\n(Custom track)')
_show(axes[2], _dual,     '(c) CLAHE Dual Channel\n(Fine-Tune track)')
_show(axes[3], _cut,      '(d) RandomCutout\n(Custom augmentation)')
plt.suptitle('Visual Example of Preprocessing Transforms', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'outputs', 'transforms_example.png'), dpi=150,
            bbox_inches='tight')
plt.show()
print('Transforms figure saved.')